# Step 1 — saved-prediction closeout analysis

This notebook analyzes the completed `v2_controlled_diagnostics` results without training or loading LaBSE. It tests whether aligned EEG helps the gated model on low-confidence or otherwise text-hard sentences more than shuffled, noise, and zero controls. It also summarizes prediction flips by class, sentence length, and reader coverage. All outputs are saved under the versioned `v3_closeout_analysis` Drive folder.

## 1. Drive and repository

Mount Drive, then clone the repository or pull its newest code. This phase uses Colab's existing analysis libraries and performs no model or package download.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/parmisbathayan/zuco-multimodal-sentiment.git'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR

## 2. Source and output paths

The source folder contains the 30 completed controlled setup/seed files. The feature cache is read only to recover the number of available readers for every sentence. No cached feature or source-result file is changed.

In [ ]:
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
FEATURES_DIR = (
    f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment/'
    'eeg_features/classical_v2_normalized_line_length'
)
RESULTS_BASE = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment'
SOURCE_RUN_DIR = f'{RESULTS_BASE}/v2_controlled_diagnostics'
ANALYSIS_TAG = 'v3_closeout_analysis'
ANALYSIS_DIR = f'{RESULTS_BASE}/{ANALYSIS_TAG}'

for path in [LABELS_CSV, FEATURES_DIR, SOURCE_RUN_DIR]:
    assert os.path.exists(path), f'Missing required path: {path}'
print('source:', SOURCE_RUN_DIR)
print('output:', ANALYSIS_DIR)

## 3. Run the closeout analysis

Run the small analysis tests, then build per-sentence diagnostics, paired aligned-control comparisons, sentence-cluster bootstrap intervals, prediction-flip summaries, plots, and a decision file. Re-running this cell safely rebuilds the same derived outputs. It does not retrain any model.

In [ ]:
!python -m unittest discover -s tests -p 'test_closeout_analysis.py'

!python analyze_results.py \
    --source-run-dir "$SOURCE_RUN_DIR" \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --analysis-tag "$ANALYSIS_TAG" \
    --bootstrap-samples 2000 \
    --minimum-delta 0.015

## 4. Read the important outputs

The findings file gives the whole-dataset comparisons, predefined text-hard subset results, prediction flips, and the stop decision. The figures show aligned-minus-control effects and favorable versus unfavorable EEG-induced flips.

In [ ]:
from IPython.display import Image, Markdown, display

findings_path = f'{ANALYSIS_DIR}/tables/findings.md'
decision_path = f'{ANALYSIS_DIR}/tables/decision.json'
display(Markdown(open(findings_path).read()))
print(open(decision_path).read())
for name in ['text_hard_subset_deltas.png', 'prediction_flips.png']:
    display(Image(f'{ANALYSIS_DIR}/plots/{name}'))